In [ ]:
# ── 저장소 루트로 이동: 노트북 위치와 무관하게 data/·outputs/ 상대경로 유지 ──
import os
from pathlib import Path
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / 'README.md').exists() and (_c / '.gitignore').exists():
        os.chdir(_c); break


# 10. 집계구 위험도 — 경계 + 성·연령별 인구(실측) + 도달지연

## 이 노트북이 하는 일
**집계구 경계**(부산 동구 78개, 대상 5동)에 **집계구별 성·연령별 인구 실측**(75+/80+)을 조인하고, 안전센터 도달지연과 곱해 **집계구 위험도**를 계산한다. 이것이 100m 배분(추정)을 대체하는 실측 기반 위험지도.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 집계구인가:** 100m 격자는 연령을 '배분(추정)'해야 하지만, 집계구는 ~500명 단위 **실측**이라 산복도로 상/하단을 실제로 구분한다. 대상지 78개 → 해상도 충분. (예: 초량3동은 동 전체론 젊지만, 그 안 고령 밀집 집계구를 집계구가 콕 집어냄)
- **왜 조인 키가 TOT_OA_CD인가:** 경계 SHP와 통계 CSV를 잇는 유일한 공통 식별자(14자리 집계구코드).
- **왜 위험도 = 75+ × 도달지연 인가 (A안):** AED는 목격된 심정지만 살리므로 독거(목격 불가)는 AED 배치 가중에서 분리. 발생확률(75+ 실측) × 도달지연만 곱한다. 독거는 별도 지표(추후).
- **왜 5186으로 변환하나:** SGIS는 EPSG:5179(UTM-K). 그래프·거리계산은 5186이라 통일.
- **왜 결측을 0으로 두나:** 5명 미만 값은 통계 비밀보호로 빠짐(NaN). 0 처리 → 약간의 과소집계(한계).

## 데이터 출처
- 집계구 경계: SGIS `bnd_oa_21030_2025_2Q`(2025 2분기). 성·연령별 인구: SGIS `21030_2024년_성연령별인구`(집계구 단위, in_age 5세). 그래프: 06.

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, glob
import geopandas as gpd, networkx as nx, osmnx as ox
CRS_M = 5186
os.makedirs("outputs", exist_ok=True)
DATA_OA  = "data/input/집계구 경계 자료"                                                       # 집계구 경계 폴더
DATA_STAT = "data/input/집계구 인구 통계자료"                                              # 집계구 성·연령별 인구 폴더

## 1. 집계구 경계 로드 + 대상 5동 필터 (5186 변환)

In [ ]:
shp = glob.glob("data/**/bnd_oa_*.shp", recursive=True)[0]                                          # 집계구 경계 SHP 자동 탐색
oa = gpd.read_file(shp)                                                               # 읽기 (EPSG:5179)
print("동구 전체 집계구:", len(oa), "| CRS:", oa.crs, "| 컬럼:", list(oa.columns))

DONGS={"21030510":"초량1동","21030520":"초량2동","21030530":"초량3동",                  # 대상 5동 (행정동코드)
       "21030550":"초량6동","21030700":"좌천동"}
oa["ADM_CD"]=oa["ADM_CD"].astype(str)                                                 # 조인·필터용 문자열화
oa["TOT_OA_CD"]=oa["TOT_OA_CD"].astype(str)                                           # 집계구코드(통계표 조인 키)
oa = oa[oa["ADM_CD"].isin(DONGS)].copy()                                              # 대상 5동만
oa["dong"]=oa["ADM_CD"].map(DONGS)                                                     # 동 이름
oa = oa.to_crs(CRS_M)                                                                 # 5186으로 통일
oa["oa_area_m2"]=oa.area                                                              # 집계구 면적(배분·검증용)
oa["cx"]=oa.geometry.centroid.x; oa["cy"]=oa.geometry.centroid.y                      # 중심 좌표(도로 스냅용)
print("대상 5동 집계구:", len(oa))
print(oa["dong"].value_counts().to_string())

## 2. 집계구 → 도달지연(안전센터 도로거리) 미리 계산\n통계와 무관하게 그래프만으로 가능 → 통계 오기 전 준비.

In [ ]:
def _sf(x):
    try: return float(x)
    except: return float("nan")
G = ox.load_graphml("outputs/graph_drive_conn.graphml",                               # 06 완전연결 그래프
    edge_dtypes={"length":float}, node_dtypes={"elev":_sf})
station_nodes=[n for n,d in G.nodes(data=True) if d.get("node_type")=="station"]       # 안전센터 노드

oa["entry_node"]=ox.distance.nearest_nodes(G, X=oa["cx"].values, Y=oa["cy"].values)     # 집계구 중심→최근접 도로노드
best={}                                                                               # 노드별 최근접 센터거리
for s in station_nodes:
    for n,d in nx.single_source_dijkstra_path_length(G, s, weight="length").items():
        if n not in best or d<best[n]: best[n]=d
oa["dist_station_m"]=oa["entry_node"].map(lambda n: best.get(n, np.nan))               # 집계구 도달지연(거리)
print("도달거리 산출:", oa["dist_station_m"].notna().sum(), "/", len(oa))
print("min/median/max: %.0f / %.0f / %.0f m" % (
    oa["dist_station_m"].min(), oa["dist_station_m"].median(), oa["dist_station_m"].max()))

## 3. 성·연령별 인구 조인 (실측 75+/80+)

통계 CSV는 **헤더 없는** long 포맷: `연도, 집계구코드(14자리), 통계항목(in_age_XXX), 값`.
65+/75+/80+를 집계해 경계에 `TOT_OA_CD`로 붙인다. 5명 미만은 결측(NaN)→0.

In [ ]:
AGE = {"p65":range(14,22),   # 65세+ = in_age_014~021
       "p75":range(16,22),   # 75세+ = in_age_016~021 (주 지표)
       "p80":range(17,22)}   # 80세+ = in_age_017~021

f = glob.glob("data/**/*성연령별인구*.csv", recursive=True)[0]                                                # 성·연령별 인구 CSV
for enc in ["cp949","euc-kr","utf-8-sig","utf-8"]:                                    # 헤더 없음 → 컬럼명 직접 부여
    try:
        df = pd.read_csv(f, header=None, names=["year","oa","item","val"], encoding=enc); break
    except Exception: continue
df["oa"]=df["oa"].astype(str); df["val"]=pd.to_numeric(df["val"], errors="coerce")     # 코드 문자열화 / 값 숫자화
print("통계 레코드:", len(df), "| 고유 집계구:", df["oa"].nunique(),
      "| 결측(<5) 비율: %.1f%%" % (df["val"].isna().mean()*100))

for name, rng in AGE.items():                                                         # 65/75/80+ 각각 집계
    codes=[f"in_age_{i:03d}" for i in rng]
    s=df[df["item"].isin(codes)].groupby("oa")["val"].sum()                           # 집계구별 합
    oa[name]=oa["TOT_OA_CD"].map(s).fillna(0)                                         # 경계에 조인(결측 0)

print("매칭:", oa["TOT_OA_CD"].isin(set(df["oa"])).sum(), "/", len(oa))
print("초량·좌천 실측 — 65+: %d / 75+: %d / 80+: %d" % (oa["p65"].sum(), oa["p75"].sum(), oa["p80"].sum()))
print("\n[동별 75+ 실측]"); print(oa.groupby("dong")["p75"].sum().astype(int).to_string())

## 4. 시점 최신화 (하이브리드) + 위험도 계산

**하이브리드:** 집계구는 2024 기준이라, jumin 2026-06 동별 75+ 총량으로 **동별 스케일**만 곱해 최신화한다(집계구 내부 분포는 유지). 이러면 **2026-06 시점 + 집계구 정밀**을 동시에 얻고, 집계구 5명미만 결측(과소집계)도 함께 보정된다.

위험도 = 정규화(75+ 최신) × 정규화(도달지연). 독거는 분리(A안).

In [ ]:
import re
# jumin 2026-06 동별 75+ 총량 (5세 버킷 합)
jf=glob.glob("data/**/*연령별인구현황*.csv", recursive=True)[0]
for enc in ["cp949","euc-kr","utf-8-sig","utf-8"]:
    try: j=pd.read_csv(jf, encoding=enc); break
    except: continue
buckets=["75~79세","80~84세","85~89세","90~94세","95~99세","100세 이상"]                # 75세 이상 5세 구간
cols75=[c for c in j.columns if any(b in c for b in buckets) and "_남_" not in c and "_여_" not in c]  # 성별 무관 거주자
for c in cols75: j[c]=j[c].map(lambda x: int(str(x).replace(",","")) if str(x).replace(",","").isdigit() else 0)
j["ju75"]=j[cols75].sum(axis=1)
name2={"초량제1동":"초량1동","초량제2동":"초량2동","초량제3동":"초량3동","초량제6동":"초량6동","좌천동":"좌천동"}
j["dong"]=j["행정구역"].apply(lambda s: next((v for k,v in name2.items() if k in str(s)), None))
ju75=j[j["dong"].notna()].groupby("dong")["ju75"].sum()                                # 동별 2026-06 75+

# 동별 스케일 = jumin2026 / 집계구2024, 집계구 p75에 곱해 최신화
oa24=oa.groupby("dong")["p75"].sum()
scale={d: (ju75[d]/oa24[d] if d in ju75 and oa24[d]>0 else 1.0) for d in oa24.index}
oa["p75_2026"]=oa.apply(lambda r: r["p75"]*scale.get(r["dong"],1.0), axis=1)            # 최신화된 집계구 75+
print("[동별 스케일 (집계구2024 → jumin2026)]")
for d in sorted(scale): print(f"  {d}: x{scale[d]:.3f}")

def nrm(s):
    s=pd.to_numeric(s,errors="coerce")
    return (s-s.min())/(s.max()-s.min()) if s.max()>s.min() else s*0
oa["f_age"]  = nrm(oa["p75_2026"])                                                     # 발생확률 = 75+ 최신(2026-06)
# 도달거리 이상치(일방통행 far-corner 우회로 7km 등 소수) 상한 클립 → 정규화 왜곡 방지
reach_cap = oa["dist_station_m"].clip(upper=oa["dist_station_m"].quantile(0.95))       # 95백분위 초과는 '매우 멂'으로 캡
oa["f_reach"]= nrm(reach_cap)                                                          # 도달지연 정규화(이상치 완화)
oa["risk"]   = oa["f_age"] * oa["f_reach"]                                             # 위험도 = 곱(A안)
oa["risk_norm"]=nrm(oa["risk"])
print("\n[위험도 상위 집계구 8개]")
print(oa.nlargest(8,"risk_norm")[["dong","TOT_OA_CD","p75_2026","dist_station_m","risk_norm"]].round(2).to_string(index=False))
print("\n[동별 평균 위험도]")
print(oa.groupby("dong")["risk_norm"].mean().sort_values(ascending=False).round(3).to_string())

## 5. 저장 + 위험지도

In [ ]:
keep=["TOT_OA_CD","ADM_CD","dong","oa_area_m2","cx","cy","entry_node",
      "dist_station_m","p65","p75","p80","p75_2026","f_age","f_reach","risk","risk_norm","geometry"]
oa[keep].to_parquet("outputs/oa_risk.parquet")                                         # 집계구 위험도(실측·2026-06 최신)
try: oa[keep].to_file("outputs/oa_risk.gpkg", driver="GPKG")
except Exception as e: print("gpkg 경고:", e)

import folium
gw=oa.to_crs(4326)
def rcol(v):
    if pd.isna(v): return "#ddd"
    t=float(v); r=int(255*min(t+0.1,1)); g=int(200*(1-t)); b=int(60*(1-t)); return f"#{r:02x}{g:02x}{b:02x}"
m=folium.Map(location=[35.122,129.045], zoom_start=14, tiles="cartodbpositron")
for _,r in gw.iterrows():
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=lambda x,c=rcol(r["risk_norm"]):{"color":"#555","weight":0.4,"fillColor":c,"fillOpacity":0.6},
        tooltip=f'{r["dong"]} 75+={int(r["p75"])} 거리={int(r["dist_station_m"])}m risk={round(r["risk_norm"],2)}').add_to(m)
legend=('<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px 12px;'
        'border:1px solid #999;font-size:12px"><b>집계구 위험도</b> (75+실측 × 도달지연)<br>'
        '<span style="color:#dd2222">&#9644;</span> 높음 &nbsp; <span style="color:#22aa22">&#9644;</span> 낮음</div>')
m.get_root().html.add_child(folium.Element(legend))
m.save("outputs/oa_risk_map.html")
print("저장: outputs/oa_risk.* + oa_risk_map.html")
print("\n[다음] 축4(공백)·축6(MCLP)를 이 집계구 위험도 기준으로 재실행")
m